# 01 · Data Loading & Cleaning
Phase 1: Load all raw data, normalize team names, validate, build team registry.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from src.data_loader import load_all
np.random.seed(42)


## 1.1 Load All Datasets

In [2]:
data = load_all(verbose=True)
results    = data['results_primary']
results_full = data['results_full']
fixtures   = data['group_fixtures']
ko_slots   = data['knockout_slots']
rankings   = data['fifa_rankings']
shootouts  = data['shootouts']
wc_stats   = data['wc_stats']
registry   = data['team_registry']
former     = data['former_names']
print("\nTeam registry preview:")
registry.head(10)


INFO | Loading results from /Users/saurabhgupta/Desktop/ml_projects/fifa-wc-2026-prediction/data/raw/results.csv


INFO |   Raw rows: 49,477


WARNING |   Found 2 duplicate match entries — dropping


INFO |   Full dataset: 49,475 rows | Primary (1990+): 32,358


INFO | former_names.csv columns: ['current', 'former', 'start_date', 'end_date']


...

INFO | Loaded FIFA rankings for 48 teams


INFO | Loaded WC historical stats: 964 matches (1930-2022)


INFO | Saved team_registry.csv: 48 teams → /Users/saurabhgupta/Desktop/ml_projects/fifa-wc-2026-prediction/data/processed/team_registry.csv



DATA LOADING COMPLETE
  results (full):        49,475 rows
  results (1990+):       32,358 rows
  group fixtures:            72 matches
  knockout slots:            32 matches
  FIFA rankings:             48 teams
  shootouts:                678 records
  WC historical stats:      964
  team registry:             48 teams


Team registry preview:


,team_name,group,confederation,conf_weight,fifa_ranking,fifa_points,is_host,is_placeholder
0,Czechia,A,UEFA,1.00,39,1505.74,False,False
1,Mexico,A,CONCACAF,0.85,14,1687.48,True,False
2,South Africa,A,CAF,0.85,60,1428.38,False,False
3,South Korea,A,AFC,0.85,25,1591.63,False,False
4,Bosnia and Herzegovina,B,UEFA,1.00,64,1387.22,False,False
5,Canada,B,CONCACAF,0.85,30,1559.48,True,False
6,Qatar,B,AFC,0.85,57,1450.31,False,False
7,Switzerland,B,UEFA,1.00,19,1650.06,False,False
8,Brazil,C,CONMEBOL,1.00,6,1765.86,False,False
9,Haiti,C,CONCACAF,0.85,83,1293.10,False,False


## 1.2 Validate Results Dataset

In [3]:
print("Results shape:", results.shape)
print("Date range:", results['date'].min().date(), "→", results['date'].max().date())
print("Unique home teams:", results['home_team'].nunique())
print("Unique away teams:", results['away_team'].nunique())
print("\nMissing values:")
print(results.isnull().sum()[results.isnull().sum() > 0])
print("\nTournament types (top 15):")
print(results['tournament'].value_counts().head(15))


Results shape: (32358, 12)
Date range: 1990-01-12 → 2026-06-27
Unique home teams: 318
Unique away teams: 310

Missing values:
home_score    24
away_score    24
dtype: int64

Tournament types (top 15):
tournament
Friendly                                10828
FIFA World Cup qualification             6977
UEFA Euro qualification                  2114
African Cup of Nations qualification     1865
UEFA Nations League                       658
African Cup of Nations                    642
AFC Asian Cup qualification               632
FIFA World Cup                            624
CFU Caribbean Cup qualification           508
CECAFA Cup                                422
CONCACAF Nations League                   422
Gold Cup                                  420
Island Games                              384
Copa América                              378
COSAFA Cup                                354
Name: count, dtype: int64


## 1.3 Group Fixtures Validation

In [4]:
print("Group fixtures shape:", fixtures.shape)
print("\nGroups and team counts:")
print(fixtures.groupby('group')[['home_team','away_team']].nunique())
print("\nAll WC 2026 teams:")
all_teams = sorted(set(fixtures['home_team']) | set(fixtures['away_team']))
print(f"  {len(all_teams)} teams: {all_teams}")


Group fixtures shape: (72, 6)

Groups and team counts:
       home_team  away_team
group                      
A              4          4
B              4          4
C              4          4
D              4          4
E              4          4
F              4          4
G              4          4
H              4          4
I              4          4
J              4          4
K              4          4
L              4          4

All WC 2026 teams:
  48 teams: ['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Brazil', 'Cabo Verde', 'Canada', 'Colombia', 'Congo DR', 'Croatia', 'Curaçao', 'Czechia', "Côte d'Ivoire", 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Tü

## 1.4 Team Registry Summary

In [5]:
print("Registry shape:", registry.shape)
print("\nBy confederation:")
print(registry.groupby('confederation').agg(
    count=('team_name','count'),
    avg_rank=('fifa_ranking','mean'),
    avg_points=('fifa_points','mean')
).round(1))

print("\nHost nations:")
print(registry[registry['is_host']])

print("\nFIFA Rankings — Top 10 WC teams:")
print(registry.nsmallest(10,'fifa_ranking')[['team_name','group','confederation','fifa_ranking','fifa_points']])


Registry shape: (48, 8)

By confederation:
               count  avg_rank  avg_points
confederation                             
AFC                9      41.9      1513.8
CAF               10      40.3      1521.8
CONCACAF           6      43.3      1507.5
CONMEBOL           6      16.5      1686.2
OFC                1      85.0      1275.6
UEFA              16      20.7      1662.3

Host nations:
   team_name group confederation  conf_weight  fifa_ranking  fifa_points  \
1     Mexico     A      CONCACAF         0.85            14      1687.48   
5     Canada     B      CONCACAF         0.85            30      1559.48   
15       USA     D      CONCACAF         0.85            17      1671.23   

    is_host  is_placeholder  
1      True           False  
5      True           False  
15     True           False  

FIFA Rankings — Top 10 WC teams:
      team_name group confederation  fifa_ranking  fifa_points
37    Argentina     J      CONMEBOL             1      1876.12
30        Spa

## 1.5 Shootouts Validation

In [6]:
print("Shootouts shape:", shootouts.shape)
print("Date range:", shootouts['date'].min().date(), "→", shootouts['date'].max().date())
print("\nSample:")
shootouts.head(5)


Shootouts shape: (678, 5)
Date range: 1967-08-22 → 2026-06-06

Sample:


,date,home_team,away_team,winner,first_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN
1,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN
2,1972-05-07,South Korea,Iraq,Iraq,NaN
3,1972-05-17,Thailand,South Korea,South Korea,NaN
4,1972-05-19,Thailand,Cambodia,Thailand,NaN


## 1.6 WC Historical Stats Preview

In [7]:
if wc_stats is not None:
    print("WC historical stats shape:", wc_stats.shape)
    print("Columns:", wc_stats.columns.tolist()[:15])
    print("Years covered:", wc_stats['Year'].min(), "→", wc_stats['Year'].max())
    print("\nCards distribution:")
    if 'home_yellow_card_long' in wc_stats.columns:
        print("  Yellow card data available")
    if 'home_red_card' in wc_stats.columns:
        rc_h = wc_stats['home_red_card'].fillna('').astype(str).str.count('·')
        rc_a = wc_stats['away_red_card'].fillna('').astype(str).str.count('·')
        print(f"  Red cards per match: { (rc_h.sum() + rc_a.sum()) / len(wc_stats) :.3f}")
else:
    print("WC historical stats not available — using empirical priors")


WC historical stats shape: (964, 44)
Columns: ['home_team', 'away_team', 'home_score', 'home_xg', 'home_penalty', 'away_score', 'away_xg', 'away_penalty', 'home_manager', 'home_captain', 'away_manager', 'away_captain', 'Attendance', 'Venue', 'Officials']
Years covered: 1930 → 2022

Cards distribution:
  Yellow card data available
  Red cards per match: 0.118


## ✅ Phase 1 Complete
All data loaded and validated. Processed files saved to `data/processed/`.